# Fast Flux Drive Workflow

## Description

!Flux-pulse control experiments for the qubit's own flux/`th_res` line: flux-amplitude calibration, flux-pulse timing check, T1 decay under flux detuning, and effective temperature/T1 vs. flux amplitude.

!This notebook is purely diagnostic: it loads the QPU tuned by the tune-up notebook and does not update or re-save it.

This is one of four template notebooks that together replace the original
monolithic workflow. **This notebook covers Flux drive
experiments** for a single qubit using the LabOne Q workflow architecture:

1. [Flux Amplitude Calibration](#flux-amplitude-calibration)
2. [Flux Pulse Check](#flux-pulse-check)
3. [Decay with Different Flux Detuning](#decay-with-different-flux-detuning)
4. [Population with Flux Drive](#population-with-flux-drive)
5. [T1 and Population vs. Flux Simultaneously](#t1-and-population-vs-flux-simultaneously)

The other three template notebooks under `qubit_thermometry/templates/` load that saved QPU
and continue from there:

- Tune-up experiments.
- Population and temperature measurements (single-shot 0/1, population sweeps).
- SINIS Calibration, Statistic and Temperature Sweep measurements.


## Changelog

All changes must be logged here! Do not make changes to this notebook other than expanding or modifying the standard experiments in a meaningful! All device-specific parameters and measurements have to be done in a different notebook!

Stick to semantic versioning:
- **Last Digit**: Fix a bug that does not add any new features and is backwards compatible -> increment last digit by one (e.g. 0.0.4 to 0.0.5)

- **Middle Digit**: Add a new feature while keeping backwards compatibility (older versions can still be executed without any modifications) -> increment middle digit by one and set last digit to 0 (e.g. 1.2.12 to 1.3.0)

- **First Digit**: Make changes that will break older versions, making them incompatible with the new version -> increment first digit by one and set all other digits to 0 (e.g. 3.2.9 to 4.0.0)

**Current Version:** <font color="#008000">**v2.0.0**</font>

**2026-09-27**, *Kha*: Updating from version v1.7.3, separated specified script for different experiments.                                                  (v2.0.0)

## Setup

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path("/Users/khatran/Desktop/qubit-thermometry_v2.0")
sys.path.insert(0, str(REPO_ROOT / "qubit_thermometry" / "src"))

In [ ]:
from copy import deepcopy

import numpy as np
from laboneq.simple import dsl, pulse_library

from qubit_thermometry.helper import data_io, qpu_io, setup
from qubit_thermometry.helper.temperature import collect_population_from_result, pop_label_list
from qubit_thermometry.workflow.analysis import fast_flux as fast_flux_analysis
from qubit_thermometry.workflow.experiments.fast_flux_decay import (
    fast_flux_decay_experiment_workflow,
)
from qubit_thermometry.workflow.experiments.flux_amplitude_calibration import (
    flux_amplitude_calibration_experiment_workflow,
)
from qubit_thermometry.workflow.experiments.population_flux import (
    population_flux_experiment_workflow,
)

`DeviceSetup` (physical instrument connections/ports/IPs) is a separate concern from the QPU (qubit calibration parameters) and is never saved/loaded -- it is cheap and deterministic to rebuild, so it is re-entered here just like the sample/qubit/cooldown identifiers below. The qubits returned by `build_device_setup` are fresh/default ones and are discarded; the actual tuned qubits come from the QPU saved by the tune-up notebook.

In [ ]:
device_setup, _default_qubits = setup.build_device_setup(
    shfqc_device_id="",  # TODO
    number_of_qubits=1,  # TODO
    ip_address="localhost",  # TODO
)
session = setup.connect_session(device_setup)

Sample/qubit/cooldown identifiers, used to locate the data directory and the tuned QPU saved by the tune-up notebook.

In [ ]:
sample_name = ""  # TODO
qubit_name = ""  # TODO
cooldown_start_date = ""  # TODO

In [ ]:
from functools import partial

data_root_directory, qpu_file_path = setup.init_data_saving(
    sample_name, cooldown_start_date, qubit_name
)
get_path_to_file = partial(setup.get_path_to_file, directory=data_root_directory)

qpu = qpu_io.load_qpu(qpu_file_path)
qubits = qpu.quantum_elements
qops = qpu.quantum_operations

folder_store, logging_store = setup.setup_logbook(data_root_directory)

In [ ]:
qubit_to_measure = qubits[0]

## Flux Amplitude Calibration

#### Experiment Parameters

In [ ]:
# baseline flux/th_res oscillator frequency (old qubit_parameters["th_res_freq"])
flux_oscillator_frequency = 0.0

n_avg_exponent = 12

flux_amp_min = 0.0
flux_amp_max = 1.0
n_flux_amp = 101
flux_amplitudes = np.linspace(flux_amp_min, flux_amp_max, n_flux_amp)

# delay after the x180 pulse, before readout
delay_time = 1e-7

# detuning sweep of the qubit drive relative to resonance_frequency_ge. The maximum
# reachable detuning is limited to the drive oscillator's usable bandwidth around
# drive_lo_frequency (a few hundred MHz).
detuning_min = 0.0
detuning_max = 300e6
n_detuning = 61
detuning_sweep = np.linspace(detuning_min, detuning_max, n_detuning)

flux_pulse = pulse_library.gaussian_square(
    uid="flux_pulse", length=2.0e-6, width=1.8e-6, amplitude=1, can_compress=True
)

temporary_parameters = {}
temp_pars = deepcopy(qubit_to_measure.parameters)
temporary_parameters[qubit_to_measure.uid] = temp_pars

print("Flux pulse:            ", flux_pulse)
print("Delay after x180:      ", delay_time)
print("Detuning sweep:        ", detuning_sweep[0] * 1e-6, "...",
      detuning_sweep[-1] * 1e-6, "MHz")
print("Number of averages:    ", 2**n_avg_exponent)

#### Run Workflow

In [ ]:
options = flux_amplitude_calibration_experiment_workflow.options()
options.count(2**n_avg_exponent)

flux_freq_results_list = []

with setup.logging_disabled(folder_store, logging_store):
    for i, detuning in enumerate(detuning_sweep):
        print("Iteration:", i, "| detuning:", detuning * 1e-6, "MHz")

        exp_workflow = flux_amplitude_calibration_experiment_workflow(
            session=session,
            qpu=qpu,
            qubit=qubit_to_measure.uid,
            flux_amplitudes=flux_amplitudes,
            flux_pulse=flux_pulse,
            flux_detuning=detuning,
            flux_oscillator_frequency=flux_oscillator_frequency,
            delay_time=delay_time,
            options=options,
            temporary_parameters=temporary_parameters,
        )
        workflow_result = exp_workflow.run()
        result = workflow_result.output

        flux_freq_results_list.append(
            result[dsl.handles.result_handle(qubit_to_measure.uid, suffix="flux_amp")].data
        )

flux_freq_results_arr = np.array(flux_freq_results_list)

#### Analysis Plots

In [ ]:
fig, ax = fast_flux_analysis.plot_flux_heatmap(
    detuning_sweep * 1e-6, flux_amplitudes, flux_freq_results_arr.real,
    xlabel="Detuning, MHz", ylabel="Flux drive amplitude, a.u.",
    title=f"Flux drive calibration - {qubit_to_measure.uid}",
)

file_path = get_path_to_file("Flux_drive_calibration_real_", ".png")
fig.savefig(file_path, dpi=600, format="png", bbox_inches="tight")

In [ ]:
flux_freq_results_arr_corr = fast_flux_analysis.correct_flux_calibration(flux_freq_results_arr)

fig, ax = fast_flux_analysis.plot_flux_heatmap(
    detuning_sweep * 1e-6, flux_amplitudes, flux_freq_results_arr_corr.imag,
    xlabel="Detuning, MHz", ylabel="Flux drive amplitude, a.u.",
    title=f"Flux drive calibration, corrected - {qubit_to_measure.uid}",
)

In [ ]:
popt, pcov, flux_calib_x, flux_calib_y = fast_flux_analysis.fit_flux_calibration(
    flux_amplitudes, detuning_sweep, flux_freq_results_arr_corr,
    qubit_to_measure.parameters.resonance_frequency_ge,
)

fig, ax = fast_flux_analysis.plot_flux_calibration_fit(
    flux_calib_x, flux_calib_y, popt, qubit_to_measure.uid
)

file_path = get_path_to_file("Flux_drive_calibration_fit_", ".png")
fig.savefig(file_path, dpi=600, format="png", bbox_inches="tight")

#### Update Parameters

In [ ]:
data_flux_calib = {
    "detuning_sweep": detuning_sweep,
    "flux_amplitude_sweep": flux_amplitudes,
    "flux_freq_results": flux_freq_results_arr,
    "flux_freq_results_corr": flux_freq_results_arr_corr,
    "fit_popt": popt,
    "delay_time": delay_time,
    "pulse_comment": "flux gaussian square length=2.0e-6, width=1.8e-6",
    "n_shots": 2**n_avg_exponent,
}
payload = data_io.build_mat_payload(
    qubit_to_measure, sample_name, qubit_name, cooldown_start_date, data=data_flux_calib
)
file_path = get_path_to_file("Flux_amp_sweep_and_freq_sweep_full_", ".mat")
data_io.save_mat(file_path, payload)

## Flux Pulse Check

#### Experiment Parameters

In [ ]:
n_avg_exponent = 11

flux_amp_min = 0.0
flux_amp_max = 1.0
n_flux_amp = 201
flux_amplitudes = np.linspace(flux_amp_min, flux_amp_max, n_flux_amp)

flux_width = 2e-6
delay_min = 10e-9
delay_max = 2000e-9
n_delay = 21
delay_sweep = np.linspace(delay_min, delay_max, n_delay)

# fixed detuning used for the pulse-timing check
flux_pulse_check_detuning = 250e6

print("Flux pulse width:      ", flux_width)
print("Delay sweep:           ", delay_sweep[0] * 1e9, "...", delay_sweep[-1] * 1e9, "ns")
print("Detuning:              ", flux_pulse_check_detuning * 1e-6, "MHz")
print("Number of averages:    ", 2**n_avg_exponent)

#### Run Workflow

In [ ]:
options = flux_amplitude_calibration_experiment_workflow.options()
options.count(2**n_avg_exponent)

flux_delay_results_list = []

with setup.logging_disabled(folder_store, logging_store):
    for i, delay in enumerate(delay_sweep):
        print("Iteration:", i, "| delay:", delay * 1e9, "ns")

        flux_pulse_sw = pulse_library.gaussian_square(
            uid="flux_pulse", length=flux_width + 2 * delay, width=flux_width,
            amplitude=1, can_compress=True,
        )

        exp_workflow = flux_amplitude_calibration_experiment_workflow(
            session=session,
            qpu=qpu,
            qubit=qubit_to_measure.uid,
            flux_amplitudes=flux_amplitudes,
            flux_pulse=flux_pulse_sw,
            flux_detuning=flux_pulse_check_detuning,
            flux_oscillator_frequency=flux_oscillator_frequency,
            delay_time=delay,
            options=options,
            temporary_parameters=temporary_parameters,
        )
        workflow_result = exp_workflow.run()
        result = workflow_result.output

        flux_delay_results_list.append(
            result[dsl.handles.result_handle(qubit_to_measure.uid, suffix="flux_amp")].data
        )

flux_delay_results_arr = np.array(flux_delay_results_list)

#### Analysis Plots

In [ ]:
fig, ax = fast_flux_analysis.plot_flux_heatmap(
    delay_sweep * 1e9, flux_amplitudes, flux_delay_results_arr.real,
    xlabel="Delay, ns", ylabel="Flux drive amplitude, a.u.",
    title=f"Flux pulse check - {qubit_to_measure.uid}",
)

file_path = get_path_to_file("Flux_pulse_check_", ".png")
fig.savefig(file_path, dpi=600, format="png", bbox_inches="tight")

#### Update Parameters

In [ ]:
data_flux_pulse_check = {
    "delay_sweep": delay_sweep,
    "flux_amplitude_sweep": flux_amplitudes,
    "flux_delay_results": flux_delay_results_arr,
    "flux_width": flux_width,
    "detuning": flux_pulse_check_detuning,
    "pulse_comment": "flux gaussian square length=width+2*delay, width=flux_width",
    "n_shots": 2**n_avg_exponent,
}
payload = data_io.build_mat_payload(
    qubit_to_measure, sample_name, qubit_name, cooldown_start_date, data=data_flux_pulse_check
)
file_path = get_path_to_file("Flux_amp_sweep_and_width_sweep_short_", ".mat")
data_io.save_mat(file_path, payload)

## Decay with Different Flux Detuning

#### Experiment Parameters

In [ ]:
n_avg_exponent = 12

t1_min = 0.0
t1_max = 30e-6
n_t1 = 31
t1_ff_sweep = setup.log_sweep_help(t1_min, t1_max, n_t1)

flux_amp_min = 0.0
flux_amp_max = 0.3
n_flux_amp = 41
flux_amp_sweep = np.linspace(flux_amp_min, flux_amp_max, n_flux_amp)

flux_pulse = pulse_library.const(
    uid="flux_pulse", length=2.0e-6, amplitude=1, can_compress=True
)

print("T1 flux delay sweep: ", t1_ff_sweep[0] * 1e6, "...", t1_ff_sweep[-1] * 1e6, "us")
print("Flux amplitude sweep:", flux_amp_sweep[0], "...", flux_amp_sweep[-1])
print("Number of averages:  ", 2**n_avg_exponent)

#### Run Workflow

In [ ]:
options = fast_flux_decay_experiment_workflow.options()
options.count(2**n_avg_exponent)

FF_decay_res_list = []

with setup.logging_disabled(folder_store, logging_store):
    for i, flux_amp in enumerate(flux_amp_sweep):
        print("Iteration:", i, "| flux amplitude:", flux_amp)

        exp_workflow = fast_flux_decay_experiment_workflow(
            session=session,
            qpu=qpu,
            qubit=qubit_to_measure.uid,
            delays=t1_ff_sweep,
            flux_pulse=flux_pulse,
            flux_amplitude=flux_amp,
            flux_oscillator_frequency=flux_oscillator_frequency,
            options=options,
            temporary_parameters=temporary_parameters,
        )
        workflow_result = exp_workflow.run()
        result = workflow_result.output

        FF_decay_res_list.append(
            result[dsl.handles.result_handle(qubit_to_measure.uid, suffix="ff_decay")].data
        )

FF_decay_res_arr = np.array(FF_decay_res_list)

#### Analysis Plots

In [ ]:
popt_t1_FF_arr, pcov_t1_FF_arr = fast_flux_analysis.fit_t1_vs_flux(
    t1_ff_sweep, FF_decay_res_arr, data_type="rot"
)

fig, ax = fast_flux_analysis.plot_t1_vs_flux(
    flux_amp_sweep, popt_t1_FF_arr, qubit_to_measure.uid, ylim=(0, 10)
)

file_path = get_path_to_file("T1_vs_flux_amplitude_", ".png")
fig.savefig(file_path, dpi=600, format="png", bbox_inches="tight")

In [ ]:
fig, ax = fast_flux_analysis.plot_flux_heatmap(
    flux_amp_sweep, t1_ff_sweep * 1e6, FF_decay_res_arr.imag,
    xlabel="Flux amplitude, a.u.", ylabel="Delay, us",
    title=f"Fast flux decay - {qubit_to_measure.uid}",
)

file_path = get_path_to_file("Fast_flux_decay_map_", ".png")
fig.savefig(file_path, dpi=600, format="png", bbox_inches="tight")

#### Update Parameters

In [ ]:
data_ff_decay = {
    "t1_ff_sweep": t1_ff_sweep,
    "flux_amp_sweep": flux_amp_sweep,
    "FF_decay_res_arr": FF_decay_res_arr,
    "popt_t1_FF_arr": popt_t1_FF_arr,
    "pcov_t1_FF_arr": pcov_t1_FF_arr,
    "pulse_comment": "flux const",
    "n_shots": 2**n_avg_exponent,
}
payload = data_io.build_mat_payload(
    qubit_to_measure, sample_name, qubit_name, cooldown_start_date, data=data_ff_decay
)
file_path = get_path_to_file("Fast_flux_decay_long_", ".mat")
data_io.save_mat(file_path, payload)

## Population with Flux Drive

### Single Experiment

#### Experiment Parameters

In [ ]:
n_avg_exponent = 15

flux_amplitude = 0.2

flux_pulse = pulse_library.gaussian_square(
    uid="flux_pulse",
    length=qubit_to_measure.parameters.reset_delay_length,
    width=qubit_to_measure.parameters.reset_delay_length - 2e-8,
    amplitude=flux_amplitude,
    can_compress=True,
)

print("Flux pulse:         ", flux_pulse)
print("Number of averages: ", 2**n_avg_exponent)

#### Run Workflow

In [ ]:
options = population_flux_experiment_workflow.options()
options.count(2**n_avg_exponent)

exp_workflow = population_flux_experiment_workflow(
    session=session,
    qpu=qpu,
    qubit=qubit_to_measure.uid,
    flux_pulse=flux_pulse,
    flux_amplitude=flux_amplitude,
    flux_oscillator_frequency=flux_oscillator_frequency,
    options=options,
    temporary_parameters=temporary_parameters,
)
workflow_result = exp_workflow.run()

#### Analysis Plots

In [ ]:
pop_flux_result = collect_population_from_result(
    workflow_result.output, qubit_to_measure.uid
)
for label in pop_label_list:
    print(f"{label}: {pop_flux_result[label]}")

### Flux Sweep Experiment

#### Experiment Parameters

In [ ]:
flux_amp_min = 0.0
flux_amp_max = 0.7
n_flux_amp = 41
flux_amp_sw = np.linspace(flux_amp_min, flux_amp_max, n_flux_amp)

n_avg_exponent = 15

print("Flux amplitude sweep:", flux_amp_sw[0], "...", flux_amp_sw[-1])
print("Number of averages:  ", 2**n_avg_exponent)

#### Run Workflow

In [ ]:
options = population_flux_experiment_workflow.options()
options.count(2**n_avg_exponent)

pop_flux_results = {}

with setup.logging_disabled(folder_store, logging_store):
    for i, flux_amp in enumerate(flux_amp_sw):
        print("Iteration number:", i, "| flux pulse amplitude:", flux_amp)

        flux_pulse_sw = pulse_library.gaussian_square(
            uid="flux_pulse",
            length=qubit_to_measure.parameters.reset_delay_length,
            width=qubit_to_measure.parameters.reset_delay_length - 2e-8,
            amplitude=flux_amp,
            can_compress=True,
        )

        exp_workflow = population_flux_experiment_workflow(
            session=session,
            qpu=qpu,
            qubit=qubit_to_measure.uid,
            flux_pulse=flux_pulse_sw,
            flux_amplitude=flux_amp,
            flux_oscillator_frequency=flux_oscillator_frequency,
            options=options,
            temporary_parameters=temporary_parameters,
        )
        workflow_result = exp_workflow.run()
        pop_result = collect_population_from_result(
            workflow_result.output, qubit_to_measure.uid
        )
        for label in pop_label_list:
            pop_flux_results.setdefault(label, []).append(pop_result[label])

for label in pop_label_list:
    pop_flux_results[label] = np.array(pop_flux_results[label])

#### Analysis Plots

In [ ]:
f_q, anharm, ABC, TABC, fig, ax = fast_flux_analysis.analyze_population_vs_flux(
    pop_flux_results, qubit_to_measure, flux_amp_sw
)

file_path = get_path_to_file("Teff_vs_FF_fast_", ".png")
fig.savefig(file_path, dpi=600, format="png", bbox_inches="tight")

In [ ]:
fig, ax = fast_flux_analysis.plot_population_iq(pop_flux_results, qubit_to_measure.uid, label="x0")

#### Update Parameters

In [ ]:
data_T_FF = {
    "n_shots": 2**n_avg_exponent,
    "flux_amp_sweep": flux_amp_sw,
    "f_q": f_q,
    "anharm": anharm,
    "comment": "Effective temperature vs. flux drive amplitude, temperatures in mK",
}
data_T_FF.update(pop_flux_results)
data_T_FF.update(TABC)
payload = data_io.build_mat_payload(
    qubit_to_measure, sample_name, qubit_name, cooldown_start_date, data=data_T_FF
)
file_path = get_path_to_file("Teff_vs_FF_fast_", ".mat")
data_io.save_mat(file_path, payload)

## T1 and Population vs. Flux Simultaneously

#### Experiment Parameters

In [ ]:
n_avg_exponent_pop = 15
n_avg_exponent_t1 = 12

flux_amp_min = 0.4
flux_amp_max = 0.7
n_flux_amp = 151
flux_amp_sw = np.linspace(flux_amp_min, flux_amp_max, n_flux_amp)

t1_min = 0.0
t1_max = 30e-6
n_t1 = 31
t1_ff_sweep = setup.log_sweep_help(t1_min, t1_max, n_t1)

print("Flux amplitude sweep:", flux_amp_sw[0], "...", flux_amp_sw[-1])
print("T1 flux delay sweep: ", t1_ff_sweep[0] * 1e6, "...", t1_ff_sweep[-1] * 1e6, "us")
print("Number of averages (population):", 2**n_avg_exponent_pop)
print("Number of averages (T1):        ", 2**n_avg_exponent_t1)

#### Run Workflow

In [ ]:
pop_options = population_flux_experiment_workflow.options()
pop_options.count(2**n_avg_exponent_pop)

t1_options = fast_flux_decay_experiment_workflow.options()
t1_options.count(2**n_avg_exponent_t1)

pop_flux_results = {}
FF_decay_res_list = []

with setup.logging_disabled(folder_store, logging_store):
    for i, flux_amp in enumerate(flux_amp_sw):
        print("Iteration number:", i, "| flux pulse amplitude:", flux_amp)

        # population
        flux_pulse_pop = pulse_library.gaussian_square(
            uid="flux_pulse",
            length=qubit_to_measure.parameters.reset_delay_length,
            width=qubit_to_measure.parameters.reset_delay_length - 2e-8,
            amplitude=flux_amp,
            can_compress=True,
        )
        exp_workflow_pop = population_flux_experiment_workflow(
            session=session,
            qpu=qpu,
            qubit=qubit_to_measure.uid,
            flux_pulse=flux_pulse_pop,
            flux_amplitude=flux_amp,
            flux_oscillator_frequency=flux_oscillator_frequency,
            options=pop_options,
            temporary_parameters=temporary_parameters,
        )
        workflow_result_pop = exp_workflow_pop.run()
        pop_result = collect_population_from_result(
            workflow_result_pop.output, qubit_to_measure.uid
        )
        for label in pop_label_list:
            pop_flux_results.setdefault(label, []).append(pop_result[label])

        # T1
        print("T1 measurement")
        flux_pulse_t1 = pulse_library.const(
            uid="flux_pulse", length=2.0e-6, amplitude=flux_amp, can_compress=True
        )
        exp_workflow_t1 = fast_flux_decay_experiment_workflow(
            session=session,
            qpu=qpu,
            qubit=qubit_to_measure.uid,
            delays=t1_ff_sweep,
            flux_pulse=flux_pulse_t1,
            flux_amplitude=flux_amp,
            flux_oscillator_frequency=flux_oscillator_frequency,
            options=t1_options,
            temporary_parameters=temporary_parameters,
        )
        workflow_result_t1 = exp_workflow_t1.run()
        FF_decay_res_list.append(
            workflow_result_t1.output[
                dsl.handles.result_handle(qubit_to_measure.uid, suffix="ff_decay")
            ].data
        )

for label in pop_label_list:
    pop_flux_results[label] = np.array(pop_flux_results[label])

FF_decay_res_arr = np.array(FF_decay_res_list)

#### Analysis Plots

In [ ]:
f_q, anharm, ABC, TABC, fig, ax = fast_flux_analysis.analyze_population_vs_flux(
    pop_flux_results, qubit_to_measure, flux_amp_sw
)

file_path = get_path_to_file("Decay_and_population_temperature_", ".png")
fig.savefig(file_path, dpi=600, format="png", bbox_inches="tight")

In [ ]:
popt_t1_FF_arr, pcov_t1_FF_arr = fast_flux_analysis.fit_t1_vs_flux(
    t1_ff_sweep, FF_decay_res_arr, data_type="rot"
)

fig, ax = fast_flux_analysis.plot_t1_vs_flux(flux_amp_sw, popt_t1_FF_arr, qubit_to_measure.uid)

file_path = get_path_to_file("Decay_and_population_T1_", ".png")
fig.savefig(file_path, dpi=600, format="png", bbox_inches="tight")

#### Update Parameters

In [ ]:
data_ff_decay_pop = {
    "FF_decay_delay": t1_ff_sweep,
    "flux_amp_sweep_pop": flux_amp_sw,
    "flux_amp_sweep_T1": flux_amp_sw,
    "FF_decay_res_arr": FF_decay_res_arr,
    "popt_t1_FF_arr": popt_t1_FF_arr,
    "n_shots_pop": 2**n_avg_exponent_pop,
    "n_shots_T1": 2**n_avg_exponent_t1,
    "pulse_T1_comment": "flux const",
    "pulse_pop_comment": "flux gaussian square",
    "f_q": f_q,
    "anharm": anharm,
}
data_ff_decay_pop.update(pop_flux_results)
data_ff_decay_pop.update(TABC)
payload = data_io.build_mat_payload(
    qubit_to_measure, sample_name, qubit_name, cooldown_start_date, data=data_ff_decay_pop
)
file_path = get_path_to_file("Decay_and_population_allparam_", ".mat")
data_io.save_mat(file_path, payload)